In [1]:
import sys
import os
current_path = os.path.abspath('')
data_path = '/'.join(current_path.split('/')[:-1]) + '/logs_annotated/'
script_path = '/'.join(current_path.split('/')[:-1]) + '/scripts/'
sys.path.append(script_path)
sys.path.append('/home/fkunneman/.local/lib/python3.10/site-packages/')
sys.path.append('/usr/lib/python3/dist-packages/')

In [6]:
import torch
import gc
import importlib
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline
from sentence_transformers import SentenceTransformer
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_huggingface.llms import HuggingFacePipeline
from langchain_community.vectorstores import Chroma
import chromadb
from chromadb.api.types import EmbeddingFunction, Documents, Embeddings

import instruction_agent
import generate_response

In [3]:
### initialize huggingface
with open('/home/fkunneman/hf.txt') as fin:
    hf = fin.read().strip()
os.environ["HF_TOKEN"] = hf
torch.cuda.empty_cache()
gc.collect()

0

In [ ]:
### Load judge llm
tokenizer_judge = AutoTokenizer.from_pretrained('robinsmits/Qwen1.5-7B-Dutch-Chat', use_fast=True)
tokenizer_judge.pad_token = tokenizer_judge.eos_token # Ensure padding token is set
judgemodel = AutoModelForCausalLM.from_pretrained('robinsmits/Qwen1.5-7B-Dutch-Chat', dtype=torch.bfloat16, trust_remote_code=True, device_map="cuda")
# set judge pipeline
pipe = pipeline(
    "text-generation",
    model=judgemodel,
    tokenizer=tokenizer_judge,
    return_full_text=False,
    max_new_tokens=250, # Limit the number of generated tokens for concise responses
    do_sample=False, # Enable sampling for more varied responses
    temperature=0.3 # Control creativity
)
llm = HuggingFacePipeline(pipeline=pipe)

In [ ]:
# initialize response generator
importlib.reload(sys.modules['generate_response'])
from generate_response import Turns, ResponseGenerator

turns = Turns()
turns.read_log(data_path + 'P1 conversatie paspoort annotated.csv')

rgen = ResponseGenerator(pipe)
output = rgen.correct_asr(turns.turns)
print('OUTPUT',output)

In [17]:
instructions_travel = data_path + 'opslag_inclusieve_spraakassistent_project/instructions_ov_stripped.csv'
instructions_passport = data_path + 'opslag_inclusieve_spraakassistent_project/instructions_paspoort_stripped_v2.csv'
pat = data_path + 'opslag_inclusieve_spraakassistent_project/patterns_v2.csv'
qa_travel = data_path + 'opslag_inclusieve_spraakassistent_project/Vraag_antwoord_ov_v5.csv'
qa_passport = data_path + 'opslag_inclusieve_spraakassistent_project/Vraag_antwoord_paspoort.csv'
nav = data_path + 'opslag_inclusieve_spraakassistent_project/navigation.csv'
